# Day 3: Feature Engineering
In this notebook, we build features for predicting IPL fantasy points, ensuring no data leakage by computing metrics using only data *prior* to the match being predicted.

In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv('/Users/aadif/projects/ipl-predictor/data/fantasy_points.csv')
df['date'] = pd.to_datetime(df['date'])
df = df.sort_values(by=['date', 'match_id']).reset_index(drop=True)
df.head()

,match_id,player,bat_pts,bowl_pts,field_pts,fantasy_points,season,date,venue,team1,team2,toss_winner,toss_decision,winner,team,won
0,335982,AA Noffke,5.5,25.0,0.0,30.5,2007,2008-04-18,M Chinnaswamy Stadium,Royal Challengers Bangalore,Kolkata Knight Riders,Royal Challengers Bangalore,field,Kolkata Knight Riders,Royal Challengers Bangalore,0
1,335982,AB Agarkar,0.0,79.0,6.0,85.0,2007,2008-04-18,M Chinnaswamy Stadium,Royal Challengers Bangalore,Kolkata Knight Riders,Royal Challengers Bangalore,field,Kolkata Knight Riders,Kolkata Knight Riders,1
2,335982,AB Dinda,0.0,58.0,0.0,58.0,2007,2008-04-18,M Chinnaswamy Stadium,Royal Challengers Bangalore,Kolkata Knight Riders,Royal Challengers Bangalore,field,Kolkata Knight Riders,Kolkata Knight Riders,1
3,335982,B Akhil,-2.0,0.0,0.0,-2.0,2007,2008-04-18,M Chinnaswamy Stadium,Royal Challengers Bangalore,Kolkata Knight Riders,Royal Challengers Bangalore,field,Kolkata Knight Riders,Royal Challengers Bangalore,0
4,335982,BB McCullum,151.0,0.0,8.0,159.0,2007,2008-04-18,M Chinnaswamy Stadium,Royal Challengers Bangalore,Kolkata Knight Riders,Royal Challengers Bangalore,field,Kolkata Knight Riders,Kolkata Knight Riders,1


## 1. Career Average
Player's average fantasy points across all prior matches.

In [2]:
df['career_avg'] = df.groupby('player')['fantasy_points'].transform(lambda x: x.expanding().mean().shift())
df[['player', 'date', 'fantasy_points', 'career_avg']].head(10)

,player,date,fantasy_points,career_avg
0,AA Noffke,2008-04-18,30.5,NaN
1,AB Agarkar,2008-04-18,85.0,NaN
2,AB Dinda,2008-04-18,58.0,NaN
3,B Akhil,2008-04-18,-2.0,NaN
4,BB McCullum,2008-04-18,159.0,NaN
5,CL White,2008-04-18,11.0,NaN
6,DJ Hussey,2008-04-18,7.0,NaN
7,I Sharma,2008-04-18,33.0,NaN
8,JH Kallis,2008-04-18,39.0,NaN
9,LR Shukla,2008-04-18,25.0,NaN


## 2. Last Season Average
Average fantasy points in the previous season. If a player skipped the immediate previous season, it won't be picked up by a simple shift, so we merge on `season - 1`.

In [3]:
season_avg = df.groupby(['player', 'season'])['fantasy_points'].mean().reset_index(name='season_avg')
season_avg['join_season'] = season_avg['season'] + 1
df = df.merge(season_avg[['player', 'join_season', 'season_avg']], left_on=['player', 'season'], right_on=['player', 'join_season'], how='left')
df.rename(columns={'season_avg': 'last_season_avg'}, inplace=True)
df.drop(columns=['join_season'], inplace=True)
df[['player', 'season', 'fantasy_points', 'last_season_avg']].head(10)

,player,season,fantasy_points,last_season_avg
0,AA Noffke,2007,30.5,NaN
1,AB Agarkar,2007,85.0,NaN
2,AB Dinda,2007,58.0,NaN
3,B Akhil,2007,-2.0,NaN
4,BB McCullum,2007,159.0,NaN
5,CL White,2007,11.0,NaN
6,DJ Hussey,2007,7.0,NaN
7,I Sharma,2007,33.0,NaN
8,JH Kallis,2007,39.0,NaN
9,LR Shukla,2007,25.0,NaN


## 3. Rolling 5 Average
Average fantasy points over the last 5 matches.

In [4]:
df['rolling_5_avg'] = df.groupby('player')['fantasy_points'].transform(lambda x: x.rolling(5, min_periods=1).mean().shift())
df[['player', 'date', 'fantasy_points', 'rolling_5_avg']].head(10)

,player,date,fantasy_points,rolling_5_avg
0,AA Noffke,2008-04-18,30.5,NaN
1,AB Agarkar,2008-04-18,85.0,NaN
2,AB Dinda,2008-04-18,58.0,NaN
3,B Akhil,2008-04-18,-2.0,NaN
4,BB McCullum,2008-04-18,159.0,NaN
5,CL White,2008-04-18,11.0,NaN
6,DJ Hussey,2008-04-18,7.0,NaN
7,I Sharma,2008-04-18,33.0,NaN
8,JH Kallis,2008-04-18,39.0,NaN
9,LR Shukla,2008-04-18,25.0,NaN


## 4. Home/Away Average
Split average based on whether the venue is the player's home ground. Here we use `team1` as a proxy for the home team.

In [5]:
df['is_home'] = (df['team'] == df['team1']).astype(int)
df['home_away_avg'] = df.groupby(['player', 'is_home'])['fantasy_points'].transform(lambda x: x.expanding().mean().shift())
df[['player', 'is_home', 'fantasy_points', 'home_away_avg']].head(10)

,player,is_home,fantasy_points,home_away_avg
0,AA Noffke,1,30.5,NaN
1,AB Agarkar,0,85.0,NaN
2,AB Dinda,0,58.0,NaN
3,B Akhil,1,-2.0,NaN
4,BB McCullum,0,159.0,NaN
5,CL White,1,11.0,NaN
6,DJ Hussey,0,7.0,NaN
7,I Sharma,0,33.0,NaN
8,JH Kallis,1,39.0,NaN
9,LR Shukla,0,25.0,NaN


## 5. Venue Average
Average fantasy points at this specific venue.

In [6]:
df['venue_avg'] = df.groupby(['player', 'venue'])['fantasy_points'].transform(lambda x: x.expanding().mean().shift())
df[['player', 'venue', 'fantasy_points', 'venue_avg']].head(10)

,player,venue,fantasy_points,venue_avg
0,AA Noffke,M Chinnaswamy Stadium,30.5,NaN
1,AB Agarkar,M Chinnaswamy Stadium,85.0,NaN
2,AB Dinda,M Chinnaswamy Stadium,58.0,NaN
3,B Akhil,M Chinnaswamy Stadium,-2.0,NaN
4,BB McCullum,M Chinnaswamy Stadium,159.0,NaN
5,CL White,M Chinnaswamy Stadium,11.0,NaN
6,DJ Hussey,M Chinnaswamy Stadium,7.0,NaN
7,I Sharma,M Chinnaswamy Stadium,33.0,NaN
8,JH Kallis,M Chinnaswamy Stadium,39.0,NaN
9,LR Shukla,M Chinnaswamy Stadium,25.0,NaN


## 6. Won Rate
Win rate of the player's team historically.

In [7]:
# Extract unique matches for teams
team_matches = df[['match_id', 'date', 'team', 'won']].drop_duplicates().sort_values('date')
team_matches['won_rate'] = team_matches.groupby('team')['won'].transform(lambda x: x.expanding().mean().shift())

df = df.merge(team_matches[['match_id', 'team', 'won_rate']], on=['match_id', 'team'], how='left')
df[['team', 'match_id', 'won', 'won_rate']].head(10)

,team,match_id,won,won_rate
0,Royal Challengers Bangalore,335982,0,NaN
1,Kolkata Knight Riders,335982,1,NaN
2,Kolkata Knight Riders,335982,1,NaN
3,Royal Challengers Bangalore,335982,0,NaN
4,Kolkata Knight Riders,335982,1,NaN
5,Royal Challengers Bangalore,335982,0,NaN
6,Kolkata Knight Riders,335982,1,NaN
7,Kolkata Knight Riders,335982,1,NaN
8,Royal Challengers Bangalore,335982,0,NaN
9,Kolkata Knight Riders,335982,1,NaN


## Saving updated dataset
Now that we have computed all features, we can drop NaN components (e.g. initial matches for rolling/expanding averages) or keep them for imputation later. We will save the augmented dataset.

In [8]:
# Optional: check null distributions
print(df.isnull().sum())

df.to_csv('/Users/aadif/projects/ipl-predictor/data/fantasy_points_featured.csv', index=False)
print("Saved to /Users/aadif/projects/ipl-predictor/data/fantasy_points_featured.csv")

match_id              0
player                0
bat_pts               0
bowl_pts              0
field_pts             0
fantasy_points        0
season                0
date                  0
venue                 0
team1                 0
team2                 0
toss_winner           0
toss_decision         0
winner                0
team                538
won                   0
career_avg          724
last_season_avg    8075
rolling_5_avg       724
is_home               0
home_away_avg      1339
venue_avg          8028
won_rate            718
dtype: int64
Saved to /Users/aadif/projects/ipl-predictor/data/fantasy_points_featured.csv
